# Omniverse Digital Twin I/O Function Design

這份 notebook 是 `Project-SprayLine` 與 NVIDIA Omniverse / Digital Twin 連動的工程實作規格。目標是先定義 local server 要送給 Omniverse 的輸入格式，以及 Omniverse 完成噴塗路徑最佳化後回傳給 local server 的輸出格式。

v1 版本不直接連線到真正的 Omniverse endpoint，而是先建立穩定的 I/O contract、mock request、mock callback、in-memory result store 與 sync log。後續只要把 dispatch 層接到真實 Omniverse API 即可。

## 1. Small Loop / Big Loop Overview

討論後的白板圖可以整理成兩層 feedback loop：小循環處理單次噴塗後的快速修正，大循環處理跨批次、跨時間窗的策略最佳化。

```mermaid
%%{init: {"flowchart": {"curve": "stepAfter"}}}%%
flowchart TB
    subgraph MainFlow["Main Flow / 主要流程"]
        direction LR
        LocalIn["Local Server<br/>本地伺服器：收集 runtime"] --> Omni["Omniverse<br/>數位分身：路徑最佳化"]
        Omni --> Robot["Robot<br/>機械手臂：執行噴塗路徑"]
        Robot --> Oven["Oven<br/>烘烤：固化塗層"]
        Oven --> ProductOut["Output<br/>完成品 / 後段檢查"]
    end

    subgraph SmallLoop["Small Loop / 小循環：快速製程修正"]
        direction LR
        Spray["Spray Paint<br/>噴塗狀態：流量 / 壓力 / 噴幅"] --> Thickness["Thickness Measurement<br/>膜厚量測：單件或單段回饋"]
        Thickness --> Width["Width Correction<br/>噴幅修正"]
        Width --> Filter["Filter Settings<br/>濾網 / 堵塞補償"]
        Filter --> LocalFeedback["Feedback Payload<br/>回傳本地伺服器"]
        LocalFeedback --> NextRequest["Next Omniverse Request<br/>產生下一段最佳化請求"]
    end

    subgraph BigLoop["Big Loop / 大循環：跨批次策略最佳化"]
        direction LR
        StageFeedback["T3 / T4 Feedback<br/>後段時間點 / 批次階段回饋"] --> QualityTrend["Batch History<br/>批次品質趨勢"]
        QualityTrend --> Knowledge["Optimization Knowledge<br/>最佳化知識累積"]
        Knowledge --> Strategy["Strategy Update<br/>更新權重與製程限制"]
        Strategy --> NewPlan["New Path Plan<br/>新路徑 / 新 constraints"]
    end

    Robot --> Spray
    NextRequest --> Omni
    ProductOut --> StageFeedback
    NewPlan --> Omni
```

小循環的用途是快速調整下一段 segment 或下一件 part。主要依據是厚度量測、噴幅、流量、壓力與 robot pose，因此 payload 需要帶 `loop_type = "small_loop"`、`feedback_source`、`quality_feedback` 與 `process_adjustments`。

大循環的用途是把多次小循環與批次結果累積成長週期決策，例如更新最佳化權重、烘烤後品質補償、filter 狀態修正策略、robot stand-off 約束或 cycle time 目標。

## 2. System Flow

整體資料流可分成即時 I/O 主流程、小循環與大循環：

```text
SprayLine runtime window
  -> local FastAPI server
  -> build_omniverse_input()
  -> Omniverse trajectory optimization request
  -> Omniverse digital twin simulation / optimization
  -> POST /api/omniverse/result
  -> receive_omniverse_output()
  -> in-memory result store + sync log
  -> dashboard / query function reads latest optimization result
```

- Fast feedback loop：依厚度、噴幅、流量、壓力、robot pose 快速調整下一段或下一件工件。
- Slow optimization loop：依批次品質、烘烤後結果、T3/T4 or batch-stage feedback 與長期趨勢更新 Omniverse 最佳化目標與 constraints。

v1 的重點是把 local server 與 Omniverse 之間的資料邊界固定下來：

- Local server 負責整理 SprayLine runtime window、產生 `request_id`、包裝最佳化任務。
- Omniverse 負責接收 runtime / robot pose / constraints，執行噴塗路徑最佳化。
- Omniverse 回傳 `request_id` 對應的最佳化結果，local server 用它把 request 和 result 串起來。

## 3. Core Function Contract

建議在專案中建立 `Omiverse/omniverse_bridge.py`，並提供三個核心 function。

| Function | Purpose | Input | Output |
|---|---|---|---|
| `build_omniverse_input(runtime_window, diagnostics=None, loop_type="small_loop", feedback_source=None, quality_feedback=None, process_adjustments=None)` | 將 SprayLine runtime window 與 feedback 轉成 Omniverse request payload | `dict`, optional feedback objects | Omniverse request `dict` |
| `receive_omniverse_output(result_payload)` | 接收並驗證 Omniverse callback result | Omniverse result `dict` | normalized stored result `dict` |
| `get_optimization_result(request_id)` | 依 `request_id` 查詢最佳化結果 | `str` | result `dict` or `None` |

這三個 function 是資料交換的核心，不應直接依賴 UI。FastAPI endpoint、dashboard 或資料庫 query function 都應該透過這一層讀寫 Omniverse I/O。`loop_type` 用來標示本次請求屬於小循環快速修正，或大循環策略最佳化。

## 4. REST API Contract

v1 REST callback API 建議放在既有 FastAPI local server 中。

| Method | Path | Purpose |
|---|---|---|
| `POST` | `/api/omniverse/optimize` | 接收 SprayLine runtime window，產生 Omniverse optimization request |
| `POST` | `/api/omniverse/result` | 接收 Omniverse callback result |
| `GET` | `/api/omniverse/result/{request_id}` | 查詢指定 request 的最佳化結果 |

v1 的 `/api/omniverse/optimize` 可以先回傳 mock dispatch object，不一定要真的呼叫外部 Omniverse server。等 Omniverse endpoint URL 確認後，再把 dispatch object 改成 HTTP client call。

## 5. Image Interpretation And Engineering Mapping

白板圖中的元素可以對應到本系統的資料與 API 邊界：

| Whiteboard element | Engineering meaning | Suggested data/API mapping |
|---|---|---|
| `Omniverse` | Omniverse digital twin optimizer | `/api/omniverse/optimize` request target |
| `Robot` | robot trajectory execution | `runtime_window.robot`, `optimized_trajectory` |
| `Spray paint` | paint process result during spraying | `signals.flow_ml_min`, `signals.spray_width_mm`, `signals.thickness_um` |
| `Oven` | post-spray curing/baking stage | `feedback_source = "oven_exit"` or batch-stage feedback |
| `Thickness Measurement` | measured quality feedback | `quality_feedback.measured_thickness_um` |
| `width` | spray width correction | `process_adjustments.spray_width_mm` |
| `Filter settings` | filter or clog compensation | `process_adjustments.filter_state_correction` |
| `T3/T4` | later time point or batch-stage feedback | `loop_type = "big_loop"`, multi-segment quality history |

因此 notebook 後續的 payload 會把白板圖中的回授資訊轉成 `feedback_source`、`quality_feedback`、`process_adjustments` 與 `recommended_adjustments`。這樣 local server 不需要理解 Omniverse 內部模型，只要維持穩定的資料 contract。


## 6. Omniverse Input Payload

Input payload 對齊 `SprayLine_3/schema/sprayline-runtime-window.schema.json`，包含 runtime metadata、reference、signals 與 robot pose。

Omniverse request 額外加上任務資訊：

```json
{
  "request_id": "uuid-string",
  "source": "sprayline-local-server",
  "task_type": "trajectory_optimization",
  "loop_type": "small_loop",
  "runtime_window": {
    "window_id": "w0001",
    "line_id": "sprayline01",
    "batch_id": "batch-A",
    "segment_id": "seg-001",
    "start_time": "2026-05-10T09:00:00.000",
    "end_time": "2026-05-10T09:00:00.200",
    "reference": {},
    "signals": {},
    "robot": {}
  },
  "diagnostics": [],
  "feedback_source": "thickness_measurement",
  "quality_feedback": {
    "measured_thickness_um": 31.0,
    "target_thickness_um": 35.0,
    "thickness_error_um": -4.0
  },
  "process_adjustments": {
    "spray_width_mm": 106.0,
    "filter_state_correction": "inspect_flow_loss"
  },
  "optimization_targets": ["thickness_stability", "collision_free", "cycle_time"],
  "constraints": {
    "position_unit": "mm",
    "pressure_unit": "bar",
    "flow_unit": "ml/min",
    "thickness_unit": "um",
    "speed_unit": "mm/s"
  }
}
```

## 7. Omniverse Output Payload

Omniverse 完成最佳化後，使用 `POST /api/omniverse/result` 回傳結果。

```json
{
  "request_id": "uuid-string",
  "status": "optimized",
  "optimized_trajectory": [
    {
      "sequence": 1,
      "tcp_x_mm": 125.0,
      "tcp_y_mm": 240.0,
      "tcp_z_mm": 380.0,
      "speed_mm_s": 420.0,
      "stand_off_mm": 175.0
    }
  ],
  "recommended_adjustments": {
    "robot_speed_mm_s": 420.0,
    "stand_off_mm": 175.0,
    "spray_width_mm": 112.0,
    "flow_ml_min": 205.0,
    "filter_state_correction": "reduce_flow_loss"
  },
  "estimated_qc_gain": 0.06,
  "collision_free": true,
  "validation_status": "pass",
  "message": "Trajectory optimized successfully.",
  "completed_at": "2026-05-25T14:30:00+08:00"
}
```

`status` 建議固定使用：

- `accepted`: Omniverse 已接受任務，但還沒完成。
- `optimized`: 已完成最佳化，且有可用軌跡。
- `failed`: 最佳化失敗，必須提供 `message` 說明原因。

`recommended_adjustments` 用來承接白板圖中的 filter / width / spray correction。小循環通常回傳下一段 robot/spray 調整；大循環則可以回傳 optimization target weight、process constraints 或模型更新建議。

## 8. Reference Python Implementation

以下 code cell 是設計參考與 mock flow。它不會修改專案程式，只示範 function contract、validation、in-memory store 與 callback result 對應方式。

In [ ]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
from uuid import uuid4
import json


REQUIRED_RUNTIME_FIELDS = [
    "window_id",
    "line_id",
    "batch_id",
    "segment_id",
    "start_time",
    "end_time",
    "reference",
    "signals",
]

ALLOWED_RESULT_STATUS = {"accepted", "optimized", "failed"}
ALLOWED_LOOP_TYPES = {"small_loop", "big_loop"}

REQUEST_STORE = {}
RESULT_STORE = {}
SYNC_LOG = []


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def append_sync_log(direction: str, request_type: str, status: str, request_id: str | None = None, error_message: str | None = None) -> dict:
    row = {
        "sync_id": str(uuid4()),
        "request_id": request_id,
        "direction": direction,
        "request_type": request_type,
        "status": status,
        "created_at": now_iso(),
        "error_message": error_message,
    }
    SYNC_LOG.append(row)
    return row


def validate_runtime_window(runtime_window: dict) -> None:
    missing = [field for field in REQUIRED_RUNTIME_FIELDS if field not in runtime_window]
    if missing:
        raise ValueError(f"runtime_window missing required fields: {missing}")


def validate_loop_payload(loop_type: str, quality_feedback: dict | list[dict] | None) -> None:
    if loop_type not in ALLOWED_LOOP_TYPES:
        raise ValueError(f"invalid loop_type: {loop_type}")
    if loop_type == "big_loop" and not quality_feedback:
        raise ValueError("big_loop request requires quality_feedback")


def build_omniverse_input(
    runtime_window: dict,
    diagnostics: list[dict] | None = None,
    loop_type: str = "small_loop",
    feedback_source: str | None = None,
    quality_feedback: dict | list[dict] | None = None,
    process_adjustments: dict | None = None,
) -> dict:
    validate_runtime_window(runtime_window)
    validate_loop_payload(loop_type, quality_feedback)

    request_id = str(uuid4())
    payload = {
        "request_id": request_id,
        "source": "sprayline-local-server",
        "task_type": "trajectory_optimization",
        "loop_type": loop_type,
        "runtime_window": deepcopy(runtime_window),
        "diagnostics": diagnostics or [],
        "feedback_source": feedback_source,
        "quality_feedback": deepcopy(quality_feedback),
        "process_adjustments": deepcopy(process_adjustments or {}),
        "optimization_targets": ["thickness_stability", "collision_free", "cycle_time"],
        "constraints": {
            "position_unit": "mm",
            "pressure_unit": "bar",
            "flow_unit": "ml/min",
            "thickness_unit": "um",
            "speed_unit": "mm/s",
        },
        "created_at": now_iso(),
    }

    REQUEST_STORE[request_id] = payload
    append_sync_log("local_to_omniverse", "trajectory_optimization", "created", request_id)
    return payload


def receive_omniverse_output(result_payload: dict) -> dict:
    request_id = result_payload.get("request_id")
    if not request_id:
        append_sync_log("omniverse_to_local", "trajectory_optimization_result", "failed", error_message="missing request_id")
        raise ValueError("result_payload missing request_id")

    status = result_payload.get("status")
    if status not in ALLOWED_RESULT_STATUS:
        append_sync_log("omniverse_to_local", "trajectory_optimization_result", "failed", request_id, f"invalid status: {status}")
        raise ValueError(f"invalid status: {status}")

    has_trajectory = bool(result_payload.get("optimized_trajectory"))
    has_adjustments = bool(result_payload.get("recommended_adjustments"))
    if status == "optimized" and not (has_trajectory or has_adjustments):
        append_sync_log("omniverse_to_local", "trajectory_optimization_result", "failed", request_id, "optimized result missing trajectory or adjustments")
        raise ValueError("optimized result must include optimized_trajectory or recommended_adjustments")

    stored = {
        "request_id": request_id,
        "status": status,
        "optimized_trajectory": result_payload.get("optimized_trajectory", []),
        "recommended_adjustments": result_payload.get("recommended_adjustments", {}),
        "optimization_strategy": result_payload.get("optimization_strategy", {}),
        "estimated_qc_gain": result_payload.get("estimated_qc_gain"),
        "collision_free": result_payload.get("collision_free"),
        "validation_status": result_payload.get("validation_status"),
        "message": result_payload.get("message", ""),
        "completed_at": result_payload.get("completed_at", now_iso()),
        "stored_at": now_iso(),
    }

    RESULT_STORE[request_id] = stored
    append_sync_log("omniverse_to_local", "trajectory_optimization_result", status, request_id)
    return stored


def get_optimization_result(request_id: str) -> dict | None:
    return RESULT_STORE.get(request_id)

## 9. Load Sample Runtime Window

這裡使用現有 sample：`SprayLine_3/samples/sprayline_runtime_w0001.json`。Notebook 放在 `Omiverse/`，所以 sample path 需要回到上一層專案根目錄。

In [ ]:
sample_candidates = [
    Path("../SprayLine_3/samples/sprayline_runtime_w0001.json"),
    Path("SprayLine_3/samples/sprayline_runtime_w0001.json"),
]
sample_path = next(path for path in sample_candidates if path.exists())
runtime_window = json.loads(sample_path.read_text(encoding="utf-8"))

pprint(runtime_window)

## 10. Build Mock Omniverse Request

這個 cell 模擬 local server 收到 runtime window 後，呼叫 `build_omniverse_input()` 產生送往 Omniverse 的 payload。

In [ ]:
diagnostics = [
    {
        "category": "nozzle_condition",
        "state_id": "nozzle_flow_loss_warning",
        "state_label": "Nozzle flow loss warning",
        "severity_rank": 2,
        "confidence": 0.86,
    }
]

omniverse_request = build_omniverse_input(runtime_window, diagnostics=diagnostics)
request_id = omniverse_request["request_id"]

pprint(omniverse_request)

## 11. Mock Omniverse Callback Result

這個 cell 模擬 Omniverse 完成最佳化後，POST 回 local server 的 result payload。`request_id` 必須與 request payload 相同，local server 才能把輸入與輸出對應起來。

In [ ]:
mock_result_payload = {
    "request_id": request_id,
    "status": "optimized",
    "optimized_trajectory": [
        {
            "sequence": 1,
            "tcp_x_mm": 125.0,
            "tcp_y_mm": 240.0,
            "tcp_z_mm": 380.0,
            "speed_mm_s": 420.0,
            "stand_off_mm": 175.0,
        },
        {
            "sequence": 2,
            "tcp_x_mm": 127.0,
            "tcp_y_mm": 244.0,
            "tcp_z_mm": 381.0,
            "speed_mm_s": 410.0,
            "stand_off_mm": 176.0,
        },
    ],
    "recommended_adjustments": {
        "robot_speed_mm_s": 420.0,
        "stand_off_mm": 175.0,
        "spray_width_mm": 112.0,
        "flow_ml_min": 205.0,
        "filter_state_correction": "reduce_flow_loss",
    },
    "estimated_qc_gain": 0.06,
    "collision_free": True,
    "validation_status": "pass",
    "message": "Trajectory optimized successfully.",
    "completed_at": now_iso(),
}

stored_result = receive_omniverse_output(mock_result_payload)
pprint(stored_result)

## 12. Query Stored Result

這個 cell 模擬 dashboard、query function 或 API endpoint 透過 `request_id` 查詢最新最佳化結果。

In [ ]:
queried_result = get_optimization_result(request_id)

assert queried_result is not None
assert queried_result["request_id"] == request_id
assert queried_result["status"] == "optimized"
assert queried_result["collision_free"] is True

pprint(queried_result)

## 13. Mock Small Loop Request And Result

小循環範例模擬厚度量測偏低。Local server 把量測結果與目前製程狀態送給 Omniverse，Omniverse 回傳下一段 robot speed、stand-off、spray width、flow 與 filter compensation 建議。

In [ ]:
small_loop_feedback = {
    "measured_thickness_um": 31.0,
    "target_thickness_um": runtime_window["reference"]["target_thickness_um"],
    "thickness_error_um": 31.0 - runtime_window["reference"]["target_thickness_um"],
    "measurement_stage": "after_oven",
}

small_loop_adjustments = {
    "spray_width_mm": runtime_window["signals"]["spray_width_mm"],
    "flow_ml_min": runtime_window["signals"]["flow_ml_min"],
    "filter_state_correction": "inspect_flow_loss",
}

small_loop_request = build_omniverse_input(
    runtime_window,
    diagnostics=diagnostics,
    loop_type="small_loop",
    feedback_source="thickness_measurement_after_oven",
    quality_feedback=small_loop_feedback,
    process_adjustments=small_loop_adjustments,
)

small_loop_result_payload = {
    "request_id": small_loop_request["request_id"],
    "status": "optimized",
    "optimized_trajectory": [
        {
            "sequence": 1,
            "tcp_x_mm": 128.0,
            "tcp_y_mm": 246.0,
            "tcp_z_mm": 379.0,
            "speed_mm_s": 405.0,
            "stand_off_mm": 174.0,
        }
    ],
    "recommended_adjustments": {
        "robot_speed_mm_s": 405.0,
        "stand_off_mm": 174.0,
        "spray_width_mm": 114.0,
        "flow_ml_min": 210.0,
        "filter_state_correction": "reduce_flow_loss",
    },
    "estimated_qc_gain": 0.05,
    "collision_free": True,
    "validation_status": "pass",
    "message": "Small-loop adjustment generated for next segment.",
    "completed_at": now_iso(),
}

small_loop_result = receive_omniverse_output(small_loop_result_payload)
assert get_optimization_result(small_loop_request["request_id"])["recommended_adjustments"]["flow_ml_min"] == 210.0
pprint(small_loop_result)

## 14. Mock Big Loop Request And Result

大循環範例模擬 T3/T4 or batch-stage feedback。Local server 將多筆 segment 品質與批次趨勢送到 Omniverse，Omniverse 回傳長週期策略，例如最佳化權重與製程 constraints。

In [ ]:
batch_quality_feedback = [
    {
        "stage": "T3",
        "batch_id": "batch-A",
        "segment_id": "seg-001",
        "avg_thickness_um": 31.5,
        "target_thickness_um": 35.0,
        "defect_count": 2,
    },
    {
        "stage": "T4",
        "batch_id": "batch-A",
        "segment_id": "seg-002",
        "avg_thickness_um": 32.2,
        "target_thickness_um": 35.0,
        "defect_count": 1,
    },
]

big_loop_request = build_omniverse_input(
    runtime_window,
    diagnostics=diagnostics,
    loop_type="big_loop",
    feedback_source="T3_T4_batch_stage_feedback",
    quality_feedback=batch_quality_feedback,
    process_adjustments={"constraint_update_reason": "post_oven_thickness_trend_low"},
)

big_loop_result_payload = {
    "request_id": big_loop_request["request_id"],
    "status": "optimized",
    "optimized_trajectory": [],
    "recommended_adjustments": {
        "target_weight_thickness_stability": 0.55,
        "target_weight_cycle_time": 0.20,
        "target_weight_collision_free": 0.25,
        "stand_off_min_mm": 170.0,
        "stand_off_max_mm": 185.0,
        "nominal_flow_ml_min": 230.0,
    },
    "optimization_strategy": {
        "strategy_scope": "batch_level",
        "apply_to_next_batch": True,
        "reason": "T3/T4 thickness trend remains below target after oven.",
    },
    "estimated_qc_gain": 0.08,
    "collision_free": True,
    "validation_status": "pass",
    "message": "Big-loop strategy update generated for next batch.",
    "completed_at": now_iso(),
}

big_loop_result = receive_omniverse_output(big_loop_result_payload)
assert big_loop_request["loop_type"] == "big_loop"
assert len(big_loop_request["quality_feedback"]) == 2
assert get_optimization_result(big_loop_request["request_id"])["optimization_strategy"]["apply_to_next_batch"] is True
pprint(big_loop_result)

## 15. Error Handling Examples

以下測試案例確認 validation error、invalid loop type、missing big-loop feedback 與 failed optimization 的處理方式。實作 FastAPI endpoint 時，這些 `ValueError` 可轉成 HTTP 400 response。

In [ ]:
# Case 1: runtime window 缺少 required field
bad_runtime_window = deepcopy(runtime_window)
bad_runtime_window.pop("signals")

try:
    build_omniverse_input(bad_runtime_window)
except ValueError as exc:
    print("Expected validation error:", exc)


# Case 2: invalid loop type
try:
    build_omniverse_input(runtime_window, loop_type="medium_loop")
except ValueError as exc:
    print("Expected loop type error:", exc)


# Case 3: big loop missing quality feedback
try:
    build_omniverse_input(runtime_window, loop_type="big_loop")
except ValueError as exc:
    print("Expected big-loop feedback error:", exc)


# Case 4: Omniverse 回傳 failed status
failed_result_payload = {
    "request_id": request_id,
    "status": "failed",
    "optimized_trajectory": [],
    "estimated_qc_gain": None,
    "collision_free": False,
    "validation_status": "fail",
    "message": "Collision-free path cannot be found under current stand-off constraints.",
    "completed_at": now_iso(),
}

failed_result = receive_omniverse_output(failed_result_payload)
pprint(failed_result)

## 16. FastAPI Endpoint Sketch

以下是後續可放入 `UserInterfaceDesign/main.py` 或獨立 router 的 endpoint sketch。這段不在 notebook 中啟動 server，只作為工程實作參考。

```python
from fastapi import FastAPI, HTTPException

@app.post("/api/omniverse/optimize")
def optimize_with_omniverse(runtime_window: dict):
    try:
        payload = build_omniverse_input(runtime_window)
        return {
            "request_id": payload["request_id"],
            "status": "created",
            "omniverse_payload": payload,
        }
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc))


@app.post("/api/omniverse/result")
def omniverse_result_callback(result_payload: dict):
    try:
        return receive_omniverse_output(result_payload)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc))


@app.get("/api/omniverse/result/{request_id}")
def read_omniverse_result(request_id: str):
    result = get_optimization_result(request_id)
    if result is None:
        raise HTTPException(status_code=404, detail="Optimization result not found")
    return result
```

## 17. Acceptance Criteria

實作完成後，應符合以下條件：

- Local server 能把 SprayLine runtime window 轉成 Omniverse request payload。
- Omniverse request 有唯一 `request_id`，且保留 runtime、reference、signals、robot pose。
- Omniverse callback result 可透過同一個 `request_id` 存取。
- `small_loop` request 可攜帶厚度量測與製程修正資訊，並回傳下一段 robot/spray adjustment。
- `big_loop` request 可攜帶 T3/T4 or batch-stage feedback，並回傳長週期 optimization strategy。
- 小循環的 `optimized` result 應包含 `optimized_trajectory` 與 `recommended_adjustments`。
- 大循環若不回傳 trajectory，必須至少提供 `recommended_adjustments` 或 `optimization_strategy`。
- `failed` result 必須保留失敗訊息，且不要求 trajectory。
- sync log 能記錄 local-to-Omniverse request 與 Omniverse-to-local result。